# DentisAI — bake-off kandidat #2: **MediaPipe Model Maker MobileNetV2 (QAT INT8)** (Kaggle / Colab, data publik saja)

Notebook ini **bukan** validasi klinis. Hasil = artefak teknis `NOT VALIDATED`. Lisensi kode & bobot MOBILENET_V2
terverifikasi (Apache-2.0, `WEIGHT_LICENSES.md`); bila sel 5 menolak dengan *"config says license_verified=true but the
gate has unverified ..."*, patch `license_gate.yaml` belum diterapkan — hentikan dan laporkan, jangan mengakali gerbang.
Kelas (ADR 014): indikasi karies gigi tetap (`caries_permanent`) / sulung (`caries_primary`); kotak hanya overlay UI.
Hanya dataset publik Ahmed 2025 (CC BY 4.0) — **jangan pernah** mengunggah foto anak/ data lokal.
Model Maker sudah *deprecated* dan butuh Python 3.10 + TensorFlow 2.15 → dipasang otomatis di venv terpisah (sel 3).

**Cara pakai (detail: `RUNBOOK.md`):** Settings → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets →
`GITHUB_TOKEN`. Lalu **Run All** dengan `SMOKE_TEST = True` (±20–30 menit); bila berhasil, ubah ke `False` dan
*Save Version → Save & Run All (Commit)*.

In [ ]:
# 0. Settings -- SATU-SATUNYA sel yang biasanya diubah
SMOKE_TEST = True           # True = uji cepat (32 foto, 1 epoch + 1 epoch QAT, seluruh rantai sampai zip unduhan). Jalankan DULU.
                            # Bila selesai tanpa error -> ubah ke False untuk pelatihan sungguhan (± 2,5-4,5 jam).
CODE_REF = "main"           # cabang/ tag/ commit repo GitHub Ryhn-F/dentisin-api yang dipakai. Biarkan "main".
EPOCHS = None               # None = nilai konfigurasi (30)
QAT_EPOCHS = None           # None = nilai konfigurasi (10). 0 = tanpa QAT -> TIDAK ada model INT8 (tidak bisa dipakai app)
BATCH_SIZE = None           # None = nilai konfigurasi (8)
TIME_BUDGET_H = 11.0        # Batas jam per sesi (Kaggle memutus di 12 jam). Model Maker TIDAK bisa melanjutkan (resume):
                            # bila waktu habis, pelatihan dihentikan dan perlu diulang dengan EPOCHS lebih kecil.
TARGET_SENSITIVITY = 0.85   # Titik operasi pada data val (teknis, bukan klinis). Biarkan.

In [ ]:
# 1. Preflight + ambil kode (jangan diubah). Gagal di sini = pesan berisi langkah perbaikan.
# >>> inlined from ml/detection/kaggle/bootstrap.py (generated by make_ipynb.py -- do not edit here)
"""Notebook bootstrap: preflight checks + getting the pipeline code. STANDARD LIBRARY ONLY.

``make_ipynb.py`` INLINES this whole file into cell 1 of every Kaggle/Colab notebook (the pipeline
code -- including ``nbkit.py`` -- is not on the machine yet when cell 1 runs), so it must stay
self-contained. Messages for the product owner are in plain Indonesian; failures raise
:class:`SystemExit` so Jupyter shows the message without a traceback.

Code is found in this order (first hit wins):

1. an attached input that contains ``ml/detection/train.py`` (e.g. the private Kaggle dataset made
   by ``artifacts.py pack-code``) -- copied to a writable scratch dir;
2. ``git clone`` of the private repo :data:`REPO_URL` at ``CODE_REF`` with a GitHub token from the
   Kaggle Secret / Colab secret / env var ``GITHUB_TOKEN``. The token is passed to git ONLY through
   ``GIT_CONFIG_*`` environment variables (never argv, never ``.git/config``, never printed) and is
   scrubbed from any git output that is shown;
3. a clear explanation of what to add.
"""

from __future__ import annotations

import base64
import contextlib
import os
import shutil
import socket
import subprocess
import sys
import time
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

BOOTSTRAP_VERSION = 1
#: nbkit.NBKIT_API must be >= this, otherwise the fetched code is older than the notebook.
REQUIRED_NBKIT_API = 5
REQUIRED_PY = (3, 11)
REPO_URL = "https://github.com/Ryhn-F/dentisin-api.git"
REPO_SLUG = "Ryhn-F/dentisin-api"
SECRET_NAME = "GITHUB_TOKEN"
CODE_MARKER = "ml/detection/train.py"
#: (host, port) pairs the notebooks need; checked by :func:`preflight`.
HOSTS_COMMON = (("github.com", 443), ("pypi.org", 443), ("files.pythonhosted.org", 443), ("zenodo.org", 443))
HOSTS_MEDIAPIPE = (("storage.googleapis.com", 443),)  # Model Maker downloads its MobileNetV2 checkpoint from here

RunFn = Callable[..., "subprocess.CompletedProcess[str]"]


def compute_env(environ: Mapping[str, str] | None = None, modules: Sequence[str] | None = None) -> str:
    env = os.environ if environ is None else environ
    mods = list(sys.modules) if modules is None else modules
    if "KAGGLE_KERNEL_RUN_TYPE" in env:
        return "kaggle"
    if "google.colab" in mods or "COLAB_RELEASE_TAG" in env or "COLAB_GPU" in env:
        return "colab"
    if any(k.startswith("KAGGLE_") for k in env):  # a stray KAGGLE_USERNAME/KAGGLE_KEY on a Colab VM must not win
        return "kaggle"
    return "local"


def default_dirs(env: str, base: Path | None = None, environ: Mapping[str, str] | None = None) -> dict[str, Path]:
    """``work`` = outputs of this run (Kaggle: /kaggle/working/outputs, saved with a committed version -- NO images
    there); ``tmp`` = scratch (code, venvs, photos, resized copies), never exported; ``inputs`` = read-only attached
    data; ``persist`` (Colab only) = a Google Drive folder that mirrors ``work`` so a dropped session can resume.
    ``work`` stays on the VM's local disk even on Colab: the trainer's stall watchdog reads log mtimes, and Drive's FUSE
    mount buffers writes. ``DENTISAI_WORK`` / ``_TMP`` / ``_INPUTS`` / ``_PERSIST`` override (local simulation)."""
    if env == "kaggle":
        dirs = {"work": Path("/kaggle/working/outputs"), "tmp": Path("/tmp/dentisai"), "inputs": Path("/kaggle/input")}
    elif env == "colab":
        dirs = {"work": Path("/content/dentisai_outputs"), "tmp": Path("/content/tmp"),
                "inputs": Path("/content/drive/MyDrive/dentisai_inputs"), "persist": Path("/content/drive/MyDrive/dentisai_runs")}
    else:
        root = base or Path.cwd() / "nb_local"
        dirs = {"work": root / "working", "tmp": root / "tmp", "inputs": root / "input"}
    e = os.environ if environ is None else environ
    for key in ("work", "tmp", "inputs", "persist"):
        if e.get(f"DENTISAI_{key.upper()}"):
            dirs[key] = Path(e[f"DENTISAI_{key.upper()}"])
    return dirs


DRIVE_ROOT = Path("/content/drive")


def mount_drive(mount: Callable[[str], Any] | None = None, root: Path = DRIVE_ROOT,
                is_mount: Callable[[str], bool] = os.path.ismount) -> bool:
    """Colab: mount Google Drive at ``root`` (Google shows its own consent pop-up). True when MyDrive is usable."""
    if is_mount(str(root)) and (root / "MyDrive").is_dir():
        return True
    try:
        if mount is None:
            from google.colab import drive

            mount = drive.mount
        mount(str(root))
    except Exception as exc:  # popup refused, not a Colab kernel, or a stale non-empty mount point
        print(f"Google Drive tidak terpasang: {type(exc).__name__}: {exc}", flush=True)
        return False
    return (root / "MyDrive").is_dir()


def resolve_dirs(env: str, *, require_persist: bool, base: Path | None = None, environ: Mapping[str, str] | None = None,
                 mount: Callable[[], bool] = mount_drive) -> dict[str, Path]:
    """:func:`default_dirs`, plus mounting Google Drive on Colab (``persist`` / Drive ``inputs``). Without Drive the
    ``persist`` key is dropped: a smoke test still runs, a real run (``require_persist``) stops with instructions --
    a dropped Colab session would otherwise lose every checkpoint."""
    dirs = default_dirs(env, base, environ)
    persist = dirs.get("persist")
    if persist is None or not persist.is_relative_to(DRIVE_ROOT) or mount():
        return dirs
    lines = ["Google Drive TIDAK terpasang, jadi checkpoint tidak bisa disimpan di luar sesi Colab ini.",
             "Jalankan ulang sel ini dan pada pop-up 'Permit this notebook to access your Google Drive files' pilih Connect to Google Drive."]
    if require_persist:
        raise fail(["BERHENTI: pelatihan sungguhan di Colab butuh Google Drive.", *lines])
    print("\n  ".join(["PERINGATAN (uji cepat):", *lines]), flush=True)
    dirs.pop("persist")
    return dirs


def rmtree(path: Path) -> None:
    """shutil.rmtree that also removes read-only files (git objects on Windows); never follows links."""
    def onexc(func: Callable[..., Any], p: str, _exc: BaseException) -> None:
        os.chmod(p, 0o700)
        func(p)

    if path.is_symlink():
        path.unlink()
    elif hasattr(os.path, "isjunction") and os.path.isjunction(path):
        os.rmdir(path)  # removes the junction only, not its target
    elif path.exists():
        isjunction = getattr(os.path, "isjunction", lambda p: False)
        for d, dirs, files in os.walk(path):  # make everything writable first; never descend into links/junctions
            dirs[:] = [n for n in dirs if not (os.path.islink(os.path.join(d, n)) or isjunction(os.path.join(d, n)))]
            for name in dirs + files:
                p = os.path.join(d, name)
                if not os.path.islink(p):
                    with contextlib.suppress(OSError):
                        os.chmod(p, 0o700)
        if sys.version_info >= (3, 12):
            shutil.rmtree(path, onexc=onexc)
        else:  # pragma: no cover - Python 3.11
            shutil.rmtree(path, onerror=lambda f, p, e: onexc(f, p, e[1]))


def _say(lines: Sequence[str]) -> str:
    bar = "=" * 78
    return "\n".join([bar, *lines, bar])


def fail(lines: Sequence[str]) -> SystemExit:
    """A plain-language stop (no traceback in Jupyter). Use as ``raise fail([...])``."""
    msg = _say(lines)
    print(msg, flush=True)
    return SystemExit(msg)


# ------------------------------------------------------------------------------------- preflight


def where(env: str, kaggle: str, colab: str, other: str = "") -> str:
    """The instruction for the platform the notebook runs on (the product owner uses both Kaggle and Colab)."""
    return {"kaggle": kaggle, "colab": colab}.get(env, other)


def python_problem(version: Sequence[int] = tuple(sys.version_info[:3]), env: str = "kaggle") -> str | None:
    if tuple(version[:2]) < REQUIRED_PY:
        have = ".".join(str(v) for v in version[:3])
        fix = where(env, "Di Kaggle: panel kanan -> Settings -> Environment -> pilih 'Always use latest environment', lalu Run All lagi.",
                    "Di Colab: Runtime -> Disconnect and delete runtime, lalu buka notebook lagi (Colab memakai Python terbaru).")
        return f"Python kernel {have} terlalu lama (butuh >= {REQUIRED_PY[0]}.{REQUIRED_PY[1]}). {fix}".strip()
    return None


def free_gb(path: Path) -> float:
    p = Path(path)
    while not p.exists() and p != p.parent:
        p = p.parent
    return shutil.disk_usage(p).free / 2**30


def ram_gb(meminfo: Path = Path("/proc/meminfo")) -> float | None:
    try:
        for line in meminfo.read_text(encoding="utf-8").splitlines():
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / 2**20
    except (OSError, ValueError, IndexError):
        return None
    return None


def gpu_names(run: RunFn = subprocess.run) -> list[str]:
    try:
        res = run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                  capture_output=True, text=True, check=False, timeout=30)
    except (OSError, subprocess.SubprocessError):
        return []
    return [ln.strip() for ln in (res.stdout or "").splitlines() if ln.strip()] if res.returncode == 0 else []


def can_reach(host: str, port: int = 443, timeout: float = 6.0, connect: Callable[..., Any] = socket.create_connection) -> bool:
    try:
        conn = connect((host, port), timeout=timeout)
    except OSError:
        return False
    with contextlib.suppress(OSError):
        conn.close()
    return True


def preflight(*, need_gpu: bool, hosts: Sequence[tuple[str, int]], disk_paths: Mapping[str, Path],
              min_free_gb: float | Mapping[str, float], env: str, run: RunFn = subprocess.run,
              reach: Callable[[str, int], bool] = can_reach, version: Sequence[int] = tuple(sys.version_info[:3])) -> dict[str, Any]:
    """Print an environment report and stop with Indonesian instructions if something blocks the run.

    ``min_free_gb`` is one number for every path or a ``{name: GB}`` mapping (Google Drive is small, the VM disk big).
    On a ``local`` machine (simulation) a missing GPU is only a warning."""
    gpus = gpu_names(run)
    reachable = {f"{h}:{p}": reach(h, p) for h, p in hosts}
    disks = {name: round(free_gb(p), 1) for name, p in disk_paths.items()}
    report: dict[str, Any] = {"python": ".".join(str(v) for v in version[:3]), "executable": sys.executable, "env": env,
                              "gpu": gpus or "TIDAK ADA", "ram_gb": None if (r := ram_gb()) is None else round(r, 1),
                              "free_disk_gb": disks, "internet": reachable}
    for k, v in report.items():
        print(f"{k:>13}: {v}")
    problems: list[str] = []
    if (pp := python_problem(version, env)) is not None:
        problems.append(pp)
    if not any(reachable.values()) and reachable:
        problems.append("Internet MATI. " + where(
            env, "Di Kaggle: panel kanan -> Settings -> Internet -> On (perlu akun dengan nomor HP terverifikasi: "
                 "kaggle.com -> Settings -> Phone verification). Lalu Run All lagi.",
            "Colab biasanya selalu online: coba Runtime -> Disconnect and delete runtime, lalu jalankan ulang."))
    elif not all(reachable.values()):
        down = ", ".join(k for k, ok in reachable.items() if not ok)
        problems.append(f"Tidak bisa menghubungi: {down}. Internet menyala tetapi situs itu tidak terjangkau -- coba lagi "
                        "beberapa menit lagi; bila tetap gagal, laporkan pesan ini.")
    if need_gpu and not gpus:
        msg = "GPU MATI. " + where(
            env, "Di Kaggle: panel kanan -> Settings -> Accelerator -> 'GPU T4 x2' (jangan P100), lalu Run All lagi. "
                 "Bila pilihan GPU tidak ada: verifikasi nomor HP dan cek sisa kuota GPU mingguan.",
            "Di Colab: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save (sesi dimulai ulang), lalu Run all. "
            "Bila T4 tidak tersedia, kuota GPU gratis hari ini habis: coba lagi nanti atau pakai Kaggle.",
            "Pasang GPU NVIDIA atau jalankan di Kaggle/Colab.")
        if env == "local":
            print("PERINGATAN (lokal):", msg)
        else:
            problems.append(msg)
    if gpus and any("P100" in g for g in gpus):
        problems.append("GPU P100 tidak didukung PyTorch/TensorFlow terbaru. Ganti ke 'GPU T4 x2' (Settings -> Accelerator).")
    need = {n: (min_free_gb.get(n, 0.0) if isinstance(min_free_gb, Mapping) else min_free_gb) for n in disks}
    low = [f"{n} ({gb} GB, butuh >= {need[n]} GB)" for n, gb in disks.items() if gb < need[n]]
    if low:
        problems.append(f"Ruang disk kurang: {', '.join(low)}. "
                        + where(env, "Mulai sesi baru (Run -> Restart) atau hapus berkas lama di /kaggle/working.",
                                "Kosongkan Google Drive (drive.google.com -> Penyimpanan) atau mulai runtime baru."))
    if problems:
        raise fail(["PERSIAPAN GAGAL -- perbaiki lalu jalankan ulang dari atas:", *[f"- {p}" for p in problems]])
    print("preflight: OK")
    return report


# ------------------------------------------------------------------------------------------ code


def get_secret(name: str = SECRET_NAME, environ: Mapping[str, str] | None = None) -> str | None:
    """Kaggle Secret -> Colab secret -> environment variable. Never logs the value."""
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        if value:
            return str(value).strip()
    except Exception:  # module absent / secret not attached to this notebook: try the next source
        pass
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return str(value).strip()
    except Exception:
        pass
    env = os.environ if environ is None else environ
    return (env.get(name) or "").strip() or None


def find_code(roots: Sequence[Path], marker: str = CODE_MARKER, max_depth: int = 4) -> Path | None:
    """Directory R such that ``R / marker`` exists (R = the folder that contains ``ml/``)."""
    for root in roots:
        if not root.is_dir():
            continue
        for depth in range(max_depth):
            for hit in sorted(root.glob("/".join(["*"] * depth + [marker]))):
                return hit.parents[len(Path(marker).parts) - 1]
    return None


def scrub(text: str, secret: str | None) -> str:
    if not secret:
        return text
    for s in (secret, base64.b64encode(f"x-access-token:{secret}".encode()).decode()):
        text = text.replace(s, "***")
    return text


def git_auth_env(token: str | None, host: str = "github.com") -> dict[str, str]:
    """Environment for git: no prompts; the token travels only in GIT_CONFIG_* (git >= 2.31), like actions/checkout."""
    # core.autocrlf=false: byte-identical checkout everywhere (sha256-pinned split files must not gain CRLF on Windows)
    env = {"GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "core.autocrlf", "GIT_CONFIG_VALUE_0": "false"}
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update({"GIT_CONFIG_COUNT": "2", "GIT_CONFIG_KEY_1": f"http.https://{host}/.extraheader",
                    "GIT_CONFIG_VALUE_1": f"AUTHORIZATION: basic {basic}"})
    return env


def classify_git_error(output: str) -> str:
    low = output.lower()
    if any(s in low for s in ("authentication failed", "could not read username", "403", "401", "invalid username or password",
                              "terminal prompts disabled")):
        return "auth"
    if "repository not found" in low or "not found" in low:
        return "not_found"
    if any(s in low for s in ("couldn't find remote ref", "unknown revision", "not our ref", "no such remote ref")):
        return "bad_ref"
    if any(s in low for s in ("could not resolve host", "failed to connect", "timed out", "network is unreachable")):
        return "network"
    return "other"


def git_clone(url: str, ref: str, dest: Path, token: str | None, *, run: RunFn = subprocess.run) -> str:
    """Shallow fetch of ``ref`` (branch, tag or full commit SHA) into a NEW ``dest``; returns the commit SHA.
    Raises RuntimeError(kind, scrubbed_output)."""
    rmtree(dest)
    dest.mkdir(parents=True)
    env = {**os.environ, **git_auth_env(token)}
    steps = [["git", "init", "-q", str(dest)],
             ["git", "-C", str(dest), "remote", "add", "origin", url],
             ["git", "-C", str(dest), "fetch", "-q", "--depth", "1", "origin", ref],
             ["git", "-C", str(dest), "checkout", "-q", "--detach", "FETCH_HEAD"],
             ["git", "-C", str(dest), "rev-parse", "HEAD"]]
    out = ""
    for cmd in steps:
        res = run(cmd, capture_output=True, text=True, env=env, check=False, timeout=600)
        out = scrub((res.stdout or "") + (res.stderr or ""), token)
        if res.returncode != 0:
            rmtree(dest)
            raise RuntimeError(classify_git_error(out), out.strip()[-2000:])
    return out.strip().splitlines()[-1] if out.strip() else ""


def token_help(env: str = "kaggle") -> list[str]:
    add_secret = where(
        env, f"  3. Kaggle (notebook ini): menu Add-ons -> Secrets -> Add Secret: Label = {SECRET_NAME}, Value = token tadi -> Save,",
        f"  3. Colab (notebook ini): ikon kunci 'Secrets' di sidebar kiri -> Add new secret: Name = {SECRET_NAME}, Value = token tadi,",
        f"  3. Simpan token di variabel lingkungan {SECRET_NAME},")
    activate = where(env, "     lalu pastikan sakelar secret itu AKTIF untuk notebook ini. Jalankan ulang sel ini.",
                     "     lalu nyalakan sakelar 'Notebook access' pada baris secret itu. Jalankan ulang sel ini.",
                     "     lalu jalankan ulang sel ini.")
    return [
        f"Cara membuat token (sekali saja, ±3 menit) -- akun GitHub yang punya akses ke repo {REPO_SLUG}:",
        "  1. github.com -> foto profil -> Settings -> Developer settings -> Personal access tokens -> Fine-grained tokens",
        "     -> Generate new token.",
        "  2. Token name: dentisai-notebook; Expiration: 30-90 hari; Resource owner: pemilik repo; Repository access:",
        f"     'Only select repositories' -> {REPO_SLUG}; Permissions -> Repository permissions -> Contents: Read-only.",
        "     (Metadata: Read-only terisi otomatis.) Generate token -> salin (diawali 'github_pat_').",
        add_secret,
        activate,
        "  Jangan menempel token di sel kode, jangan membagikan notebook dengan secret di dalamnya.",
    ]


def acquire_code(*, roots: Sequence[Path], tmp: Path, ref: str, url: str = REPO_URL, env: str = "kaggle",
                 secret: Callable[[], str | None] = get_secret, run: RunFn = subprocess.run) -> tuple[Path, dict[str, Any]]:
    """Returns (writable folder containing ``ml/``, info). See the module docstring for the order."""
    url = os.environ.get("DENTISAI_GIT_URL", url)  # mirror / local simulation override
    attached = find_code(roots)
    src = tmp / "src"
    if attached is not None:
        rmtree(src)
        shutil.copytree(attached / "ml", src / "ml", ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".venv"))
        print(f"kode: dataset terlampir {attached} (disalin ke {src})")
        return src, {"source": "attached_dataset", "path": str(attached)}
    token = secret()
    if not token:
        raise fail(["KODE TIDAK DITEMUKAN. Notebook butuh kode pipeline DentisAI. Pilih SATU:",
                    f"A) (disarankan) Tambahkan {where(env, 'Kaggle Secret', 'secret Colab', 'secret')} bernama {SECRET_NAME} "
                    "berisi token GitHub read-only.", *token_help(env),
                    where(env, "B) Atau lampirkan dataset kode privat 'dentisai-ml-code' (panel kanan -> Add Input), dibuat dengan",
                          "B) Atau salin folder kode ke Google Drive: MyDrive/dentisai_inputs/ (folder berisi ml/), dibuat dengan",
                          "B) Atau sediakan folder kode privat, dibuat dengan"),
                    "   `uv run python artifacts.py pack-code --with-processed --out dentisai-ml-code.zip` (RUNBOOK §2)."])
    print(f"kode: git clone {url} @ {ref} (token dari secret {SECRET_NAME}; tidak dicetak)")
    for attempt in range(3):
        try:
            sha = git_clone(url, ref, src, token, run=run)
            break
        except RuntimeError as exc:
            kind, detail = (*exc.args, "", "")[:2]
            if kind == "network" and attempt < 2:
                time.sleep(10 * (attempt + 1))
                continue
            why = {"auth": f"Token di secret {SECRET_NAME} ditolak GitHub (salah, kedaluwarsa, atau tidak diberi akses repo {REPO_SLUG}).",
                   "not_found": f"Repo {REPO_SLUG} tidak terlihat dengan token ini (token fine-grained harus memilih repo itu, Contents: Read).",
                   "bad_ref": f"CODE_REF = {ref!r} tidak ada di repo (cek nama cabang/tag/commit di sel 0).",
                   "network": "Jaringan ke github.com gagal berulang kali (Internet harus On)."}.get(str(kind), "git clone gagal.")
            raise fail(["GAGAL MENGAMBIL KODE DARI GITHUB.", why, *(token_help(env) if kind in ("auth", "not_found") else []),
                        "Keluaran git (token disensor):", str(detail)[-1500:]]) from None
    if not (src / CODE_MARKER).is_file():
        raise fail([f"Repo terambil (commit {sha[:12]}) tetapi {CODE_MARKER} tidak ada. CODE_REF salah? Laporkan pesan ini."])
    print(f"kode: commit {sha}")
    return src, {"source": "git", "url": url, "ref": ref, "commit": sha}


def check_nbkit_api(api: int, required: int = REQUIRED_NBKIT_API) -> None:
    if api < required:
        raise fail([f"Kode yang terambil lebih lama dari notebook ini (nbkit API {api} < {required}).",
                    "Pastikan perubahan terbaru sudah di-push ke GitHub (cabang di CODE_REF), atau buat ulang dataset kode",
                    "dengan `artifacts.py pack-code`, lalu Run All lagi."])
# <<< end of bootstrap.py
CANDIDATE = "mediapipe_modelmaker_mobilenetv2_ssd"
import json
import sys
import time
from pathlib import Path

T0 = time.time()
ENV = compute_env()
DIRS = default_dirs(ENV)
TMP = DIRS["tmp"]
WORK = DIRS["work"] / (CANDIDATE + ("_smoke" if SMOKE_TEST else ""))  # persisted outputs: NO photos, NO image arrays
for d in (TMP, WORK):
    d.mkdir(parents=True, exist_ok=True)
REPORT = preflight(need_gpu=True, hosts=(*HOSTS_COMMON, *HOSTS_MEDIAPIPE), disk_paths={"tmp": TMP, "working": DIRS["work"]},
                   min_free_gb=8.0 if SMOKE_TEST else 12.0, env=ENV)
CODE, CODE_INFO = acquire_code(roots=[DIRS["inputs"]], tmp=TMP, ref=CODE_REF)
DET = CODE / "ml" / "detection"
sys.path[:0] = [str(DET), str(DET / "kaggle")]
# Re-running this cell in a kernel that already imported an OLDER nbkit would silently keep the stale module
# (AttributeError: module 'nbkit' has no attribute ...): drop it so the freshly fetched code is used.
for _m in [m for m in sys.modules if m == "nbkit" or m.startswith("nbkit.")]:
    del sys.modules[_m]
import nbkit

check_nbkit_api(nbkit.NBKIT_API)
print(json.dumps(nbkit.env_record({"code": CODE_INFO, "smoke_test": SMOKE_TEST, "work": str(WORK)}), indent=1))

In [ ]:
# 2. Data Ahmed 2025 (otomatis: dataset terlampir -> atau unduh dari Zenodo) + anotasi kanonik + split pasien (dari repo)
missing = nbkit.missing_modules(nbkit.KERNEL_MODULES)  # small pure-Python pipeline deps (e.g. defusedxml) the kernel may lack
if missing:
    nbkit.sh([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *missing])
SPLIT_OF = json.loads((CODE / "ml/data/splits/ahmed2025/splits.json").read_text(encoding="utf-8"))["images"]
DATA = nbkit.prepare_ahmed(inputs=[DIRS["inputs"]], tmp=TMP, smoke=SMOKE_TEST, split_of=SPLIT_OF)
DATA_ROOT = Path(DATA["root"])
PROCESSED = TMP / ("processed_smoke" if SMOKE_TEST else "processed") / "ahmed2025"
info = nbkit.build_processed(src_ml=CODE / "ml", data_root=DATA_ROOT, processed=PROCESSED, images=DATA["images"])
print(json.dumps(info, indent=1))
assert set(info["boxes_per_class"]) <= {"caries_permanent", "caries_primary"}, info["boxes_per_class"]  # ADR 014: D / d
P = nbkit.processed_paths(PROCESSED)
META_CSV = WORK / "out" / "metadata.csv"
nbkit.metadata_from_splits(P["splits"], META_CSV)

In [ ]:
# 3. Pasang: venv Model Maker (Python 3.10, TF 2.15.1 + CUDA, versi terkunci) dan venv ekspor/evaluasi (Python 3.12)
UV = nbkit.uv_command()
MM_VENV = TMP / "venv-mm"
MM_PY = nbkit.ensure_venv(MM_VENV, nbkit.MM_PY, nbkit.MM_LOCK, uv=UV)
nbkit.require_imports(MM_PY, ["pkg_resources", "tensorflow_hub", "mediapipe_model_maker.python.vision.object_detector"])  # fail fast, in seconds
GPUS = nbkit.ensure_tf_gpu(MM_PY, MM_VENV, need=ENV != "local")
EXPORT_PY = nbkit.ensure_venv(TMP / "venv-export", nbkit.EXPORT_PY, nbkit.EXPORT_LOCK, uv=UV)
nbkit.sh([EXPORT_PY, "-c", "import ai_edge_litert, numpy; print('litert ok, numpy', numpy.__version__)"])

In [ ]:
# 4. Uji alat dalam beberapa menit: latih 1 epoch + 1 epoch QAT pada 8 foto, lalu ekspor (sebelum pelatihan panjang)
MINI = TMP / "mm_toolcheck"
print(nbkit.mini_layout(P["canonical"], P["splits"], DATA_ROOT, MINI, layout="mediapipe", n={"train": 8, "val": 4}))
t_check = time.time()
nbkit.sh([MM_PY, DET / "kaggle" / "mm_train_py310.py", "--train", MINI / "data/train", "--val", MINI / "data/val",
          "--export-dir", MINI / "exported", "--epochs", "1", "--batch-size", "4", "--qat-epochs", "1", "--cache-dir", MINI / "cache"])
_have = {p.name for p in (MINI / "exported").iterdir()}
assert "model_fp32.tflite" in _have, "Model Maker toolcheck produced no float model -- see log above"
assert {"model_int8_qat.tflite", "model_int8_ptq.tflite"} & _have, "Model Maker toolcheck produced no INT8 model (QAT and PTQ both failed) -- see log above"
print("INT8 method in the toolcheck:", "QAT" if "model_int8_qat.tflite" in _have else "post-training quantization (QAT failed, see warning above)")
print(f"toolcheck OK in {time.time() - t_check:.0f}s:", sorted(p.name for p in (MINI / "exported").iterdir()))

In [ ]:
# 5. Konfigurasi run + gerbang lisensi (dry run). Penolakan di sini final -- jangan mengubah gerbang dari notebook.
RUN_CFG = WORK / "cfg" / f"{CANDIDATE}.yaml"
cfg_over: dict = {
    "data": {"canonical": str(P["canonical"]), "splits": str(P["splits"]), "images_root": str(DATA_ROOT)},
    "trainer_options": {"python": str(MM_PY), "work_dir": str(TMP / ("work_smoke" if SMOKE_TEST else "work")),
                        "cache_dir": str(TMP / ("mm_cache_smoke" if SMOKE_TEST else "mm_cache"))},
}
train_over = {k: v for k, v in (("epochs", EPOCHS), ("qat_epochs", QAT_EPOCHS), ("batch_size", BATCH_SIZE)) if v is not None}
if SMOKE_TEST:
    train_over = {"epochs": 1, "qat_epochs": 1, "batch_size": 4}
if train_over:
    cfg_over["train"] = train_over
CFG = nbkit.write_run_config(DET / "configs" / f"{CANDIDATE}.yaml", RUN_CFG, cfg_over)
RUNS = WORK / "runs"
nbkit.sh([sys.executable, "train.py", "--cfg", RUN_CFG, "--dry-run", "--allow-unverified-license", "--runs-dir", RUNS], cwd=DET)

In [ ]:
# 6. Latih + QAT + ekspor (di venv Python 3.10; ±1,5-3 jam di T4; Model Maker tidak bisa resume)
TRAIN_H = nbkit.remaining_hours(TIME_BUDGET_H, T0, reserve_h=0.75)
print(f"training budget this session: {TRAIN_H} h")
CFG = nbkit.write_run_config(DET / "configs" / f"{CANDIDATE}.yaml", RUN_CFG,
                             nbkit.deep_merge(cfg_over, {"trainer_options": {"time_budget_h": TRAIN_H}}))
rc = nbkit.sh([sys.executable, "train.py", "--cfg", RUN_CFG, "--allow-unverified-license", "--runs-dir", RUNS], cwd=DET, check=False)
RUN_JSON = nbkit.latest_run(RUNS, CANDIDATE)
run = json.loads(RUN_JSON.read_text(encoding="utf-8"))
print(run["status"], json.dumps(run.get("results"), indent=1))
assert rc == 0, f"training failed: {run.get('error')} -- lihat log di atas (RUNBOOK §8)"
EXPORTED = RUN_JSON.parent / "exported"
FP32, QAT, ANCHORS = EXPORTED / "model_fp32.tflite", EXPORTED / "model_int8_qat.tflite", EXPORTED / "anchors.json"
PTQ = EXPORTED / "model_int8_ptq.tflite"
# INT8 = QAT when it worked, else post-training INT8 calibrated on training photos (Kaggle 2026-10-12: tfmot QAT
# crashed on Conv2DBNBlock). Both are uint8-in/uint8-out Model Maker exports, so the later cells are identical.
INT8_MODEL = QAT if QAT.is_file() else PTQ
INT8_METHOD = "qat" if INT8_MODEL == QAT else "ptq"
assert INT8_MODEL.is_file(), "no INT8 model: QAT and post-training INT8 both failed -- see mm_result.json export_errors above"
print({p.name: round(p.stat().st_size / 2**20, 2) for p in (FP32, INT8_MODEL)}, "INT8 method:", INT8_METHOD, "anchors:", len(json.loads(ANCHORS.read_text())))
if INT8_METHOD == "ptq":
    print("PERINGATAN: INT8 hasil post-training quantization (bukan QAT); akurasi bisa lebih rendah -- nilai lewat paritas & evaluasi di bawah.")

In [ ]:
# 7. Sampel paritas (split VAL, 50 foto; QAT tidak butuh set kalibrasi). Disimpan di TMP: tidak pernah diekspor.
PARITY = TMP / "export" / "parity_val.npy"
nbkit.sh([EXPORT_PY, "preprocess.py", "calib", "--cfg", RUN_CFG, "--split", "val", "--n", "50", "--out", PARITY], cwd=DET)

In [ ]:
# 8. Bundel dev (gerbang ukuran <= 15 MB; tensor/ kuantisasi dibaca interpreter) + paritas fp32 vs QAT (informatif:
#      QAT menyetel ulang bobot, jadi kecocokan lebih rendah dari PTQ murni adalah wajar)
EXPORTS = WORK / "exports"
VERSION_BASE = "0.1.0"
BUNDLE = EXPORTS / nbkit.next_dev_version(EXPORTS, VERSION_BASE)
nbkit.sh([EXPORT_PY, "export.py", "--cfg", RUN_CFG, "--format", "tflite", "--int8", "--tflite-in", INT8_MODEL, "--anchors", ANCHORS,
          "--out", BUNDLE, "--run-id", run["run_id"], "--allow-unverified-license"], cwd=DET)
print(json.dumps(json.loads((BUNDLE / "labels.json").read_text())["output"]["tensors"], indent=1))
PARITY_JSON = WORK / "out" / "parity.json"
nbkit.sh([EXPORT_PY, "check_int8_parity.py", "--float-model", FP32, "--int8-model", BUNDLE / "model.tflite", "--samples", PARITY,
          "--labels", BUNDLE / "labels.json", "--report", PARITY_JSON], cwd=DET, check=False)
parity = json.loads(PARITY_JSON.read_text(encoding="utf-8")) if PARITY_JSON.is_file() else {"passed": False, "failures": ["no report"]}
print("parity passed:", parity["passed"], parity["failures"])

In [ ]:
# 9. Prediksi dengan bundel INT8 (preprocess + decode app) pada val dan test; fp32 sebagai pembanding pada test
OUT = WORK / "out"
common = ["--gt", P["canonical"], "--splits", P["splits"], "--images-root", DATA_ROOT]
for split in ("val", "test"):
    nbkit.sh([EXPORT_PY, "predict.py", "--model", BUNDLE / "model.tflite", "--labels", BUNDLE / "labels.json", "--split", split,
              *common, "--out", OUT / f"preds_int8_{split}.jsonl"], cwd=DET)
nbkit.sh([EXPORT_PY, "predict.py", "--model", FP32, "--labels", BUNDLE / "labels.json", "--split", "test",
          *common, "--out", OUT / "preds_float_test.jsonl"], cwd=DET, check=False)  # fp32 I/O differs from uint8: best effort

In [ ]:
# 10. Ambang pada VAL, lalu evaluasi (val + test, CI bootstrap per kelompok pasien)
THR = OUT / "thresholds.json"
nbkit.sh([EXPORT_PY, "thresholds.py", "--pred", OUT / "preds_int8_val.jsonl", "--gt", P["canonical"], "--splits", P["splits"],
          "--target-sensitivity", str(TARGET_SENSITIVITY), "--out", THR], cwd=DET)
for split, preds in (("val", "preds_int8_val.jsonl"), ("test", "preds_int8_test.jsonl"), ("test", "preds_float_test.jsonl")):
    if not (OUT / preds).is_file():
        continue
    tag = preds.replace("preds_", "").replace(".jsonl", "")
    nbkit.sh([EXPORT_PY, "eval.py", "--gt", P["canonical"], "--splits", P["splits"], "--split", split, "--pred", OUT / preds,
              "--metadata", META_CSV, "--thresholds", THR, "--candidate", CANDIDATE, "--model-version", BUNDLE.name,
              "--report", OUT / f"report_{tag}.json"], cwd=DET)

In [ ]:
# 11. Bundel final dengan ambang + bukti; verifikasi
FINAL = EXPORTS / nbkit.next_dev_version(EXPORTS, VERSION_BASE)
nbkit.sh([EXPORT_PY, "export.py", "--cfg", RUN_CFG, "--format", "tflite", "--int8", "--tflite-in", INT8_MODEL, "--anchors", ANCHORS,
          "--out", FINAL, "--run-id", run["run_id"], "--thresholds", THR, "--eval-report", OUT / "report_int8_test.json",
          "--parity-report", PARITY_JSON, "--allow-unverified-license"], cwd=DET)
nbkit.sh([EXPORT_PY, "export.py", "verify-bundle", FINAL], cwd=DET)

In [ ]:
# 12. Catatan run + zip unduhan (TANPA gambar, TANPA .npy, TANPA checkpoint)
MEAS = OUT / "measurements_partial.json"
MEAS.write_text(json.dumps({CANDIDATE: {
    "int8_size_mb": round((FINAL / "model.tflite").stat().st_size / 2**20, 3),
    "export_notes": f"Model Maker 0.2.1.4 export_model, INT8 method={INT8_METHOD} (uint8 I/O, raw boxes+scores, anchors.json)",
    "int8_method": INT8_METHOD,
    "parity_passed": parity["passed"], "weight_license_clarity": "clear", "smoke_test": SMOKE_TEST}}, indent=1), encoding="utf-8")
(OUT / "env.json").write_text(json.dumps(nbkit.env_record({"processed": info, "code": CODE_INFO, "data_source": DATA["source"],
                                                           "smoke_test": SMOKE_TEST, "tf_gpus": GPUS}), indent=1), encoding="utf-8")
files = [RUN_JSON, THR, PARITY_JSON, MEAS, OUT / "env.json", *sorted(OUT.glob("report_*.json")), RUN_CFG,
         EXPORTED / "mm_result.json", EXPORTED / "metadata.json"]
nbkit.sh([EXPORT_PY, "artifacts.py", "collect", "--out-dir", WORK / f"download_{CANDIDATE}", "--bundle", FINAL,
          *[a for f in files if f.is_file() for a in ("--file", f)]], cwd=DET)

In [ ]:
# 13. Pemeriksaan akhir: tidak ada foto di keluaran + daftar berkas yang perlu diunduh
bad = nbkit.forbidden_outputs(DIRS["work"])
assert not bad, f"photos/arrays found in the persisted outputs (must never happen): {bad[:5]}"
print("\n".join(nbkit.describe_outputs(WORK)))
print(f"\nSelesai dalam {(time.time() - T0) / 3600:.2f} jam. SMOKE_TEST = {SMOKE_TEST}.")
if SMOKE_TEST:
    print("UJI CEPAT BERHASIL. Ubah SMOKE_TEST = False di sel 0, lalu Save Version -> Save & Run All (Commit).")
else:
    print(f"Unduh {WORK / f'download_{CANDIDATE}.zip'} (tab Output) -> backend/ml/runs/kaggle/ (RUNBOOK §6).")